In [0]:
from pyspark.sql import SparkSession

In [0]:
spark

#### Turn Off AQE

In [0]:
spark.conf.set("spark.sql.adaptive.enabled","false")

In [0]:
spark.conf.get("spark.sql.adaptive.enabled")

In [0]:
from pyspark.sql.functions import *


In [0]:
df = spark.read.format('csv').option('inferSchema',True).option('header',True).load("dbfs:/Volumes/workspace/default/file/BigMart Sales (1).csv")

In [0]:
df.display()

Spark will create logicaln partition on the above data in chunks. 128 mb default size will be partitioned for each file in 128mmb chunk of each.Below comand to see partionedno. of file.

In [0]:
df.rdd.getNumPartitions()

BELOW COMAND To change size of THE PARTION OF FILE by default 128mb. We are changing to 128kb

In [0]:
#cHANGING THE DEFAULT PARTITION SIZE TO 128 KB
spark.conf.set("spark.sql.files.maxPartitionBytes",131072)

In [0]:
df.rdd.getNumPartitions()

In [0]:
#cHANGING back to 128 mb of default size
spark.conf.set("spark.sql.files.maxPartitionBytes",134217728)

####Repartitioning 
without changing the default partition we can run partiotion using below comand

In [0]:
df=df.repartition(10)

In [0]:
df.display()

In [0]:
df.rdd.getNumPartitions()

We need partition to apply parallelism and now 10 task will be transfered to the executor through cores. 
Each executor can have so many cores. Each core can perform the parallelism. eg : 1 executor have 4 cores then 4 task at one time. Task is baically the partition we provide it to do.

In [0]:
#To see which partition hold which data we can use below comand
df.withColumn("partiotion_id",spark_partition_id()).display()

####Data Writing

In [0]:
df.write.format("parquet")\
    .mode("append")\
    .option('path','dbfs:/Volumes/workspace/default/file/parquetwrite')\
    .save()

#####New data reading

In [0]:
df_new = spark.read.format('parquet').load("dbfs:/Volumes/workspace/default/file/parquetwrite")

In [0]:
df_new.display()

In [0]:
df_new=df_new.filter(col("Outlet_Location_Type")=='Tier 1')

In [0]:
df_new.display()

**Scanning Optimization**

In [0]:
df.write.format("parquet")\
    .mode("append")\
    .partitionBy("Outlet_Location_Type")\
    .option('path','dbfs:/Volumes/workspace/default/file/parquetwriteOpt')\
    .save()


In [0]:
df_new = spark.read.format('parquet').load("dbfs:/Volumes/workspace/default/file/parquetwriteOpt")

df_new=df_new.filter(col("Outlet_Location_Type")=='Tier 1')



In [0]:
df_new.display()